In [1]:
import json
import numpy as np
import plotly.graph_objects as go

# --- 1. Your Visualization Function ---
def visualize_cameras(extrinsic_matrices, Title="", scale=200.0, labels=None):
    fig = go.Figure()

    # Lists for frustum lines
    all_x, all_y, all_z = [], [], []
    
    # Lists for camera centers (spheres) and labels
    cam_centers = []
    
    # Use provided labels if available, else default to index
    if labels is None:
        plot_labels = [f"Cam {i}" for i in range(len(extrinsic_matrices))]
    else:
        plot_labels = labels

    for i in range(len(extrinsic_matrices)):
        E = extrinsic_matrices[i]
        R = E[:3, :3]
        t = E[:3, 3]
        
        # Camera-to-World matrix
        C_to_W = np.eye(4)
        '''     C_to_W[:3, :3] = R.T  # Inverting the rotation
        C_to_W[:3, 3] = t     # Keeping the location

        C = -R.T @ t
        C_to_W[:3, 3] = C'''

        C = -R.T @ t
        
        # Camera-to-World matrix
        C_to_W = np.eye(4)
        C_to_W[:3, :3] = R.T  # Inverse of rotation
        C_to_W[:3, 3] = C     # Correct Camera Center
        cam_pos = C_to_W[:3, 3]
        cam_centers.append(cam_pos)
        
        # --- Frustum Calculation ---
        s = scale
        corners_cam = np.array([
            [0, 0, 0],          # 0: Center
            [-s, -s, s*1.5],    # 1: Top-Left
            [s, -s, s*1.5],     # 2: Top-Right
            [s, s, s*1.5],      # 3: Bottom-Right
            [-s, s, s*1.5],     # 4: Bottom-Left
        ])
        
        corners_homo = np.hstack([corners_cam, np.ones((5, 1))])
        p = (C_to_W @ corners_homo.T).T[:, :3]
        
        # Line sequence for a pyramid
        indices = [0, 1, 2, 0, 2, 3, 0, 3, 4, 0, 4, 1, 1, 2, 3, 4, 1]
        for idx in indices:
            all_x.append(p[idx, 0]); all_y.append(p[idx, 1]); all_z.append(p[idx, 2])
        all_x.append(None); all_y.append(None); all_z.append(None)

    # 1. Add Frustums
    fig.add_trace(go.Scatter3d(
        x=all_x, y=all_y, z=all_z,
        mode='lines',
        line=dict(color='rgba(100, 100, 255, 0.6)', width=2),
        name='Frustums',
        hoverinfo='none'
    ))

    # 2. Add Spheres (Markers) and Labels
    cam_centers = np.array(cam_centers)
    fig.add_trace(go.Scatter3d(
        x=cam_centers[:, 0],
        y=cam_centers[:, 1],
        z=cam_centers[:, 2],
        mode='markers+text',
        marker=dict(
            size=6,
            color=np.arange(len(extrinsic_matrices)),
            colorscale='Viridis',
            opacity=0.8
        ),
        text=plot_labels,
        textposition="top center",
        name='Camera Positions'
    ))

    # 3. Add World Origin for context
    fig.add_trace(go.Scatter3d(
        x=[0], y=[0], z=[0],
        mode='markers',
        marker=dict(size=8, color='red', symbol='cross'),
        name='World Origin'
    ))

    fig.update_layout(
        scene=dict(
            xaxis_title='X (mm)',
            yaxis_title='Y (mm)',
            zaxis_title='Z (mm)',
            aspectmode='data'
        ),
        margin=dict(l=0, r=0, b=0, t=40),
        title=f"{Title}"
    )
    
    fig.show()

# --- 2. Load Your Data ---
json_data_converted = """
{
    "calibDataSource": "converted_self_to_panoptic_v2",
    "cameras": [
        {
            "name": "00_19",
            "type": "custom",
            "resolution": [
                2304,
                2664
            ],
            "K": [
                [
                    4565.195,
                    0.0,
                    2322.638
                ],
                [
                    0.0,
                    4564.824,
                    2726.12
                ],
                [
                    0.0,
                    0.0,
                    1.0
                ]
            ],
            "distCoef": [
                -0.145,
                0.168,
                -0.0002,
                -0.0003,
                -0.062
            ],
            "R": [
                [
                    0.1373341854853676,
                    -0.03563069847352751,
                    -0.9898837178292
                ],
                [
                    0.531217490023328,
                    -0.84083241529084,
                    0.10396551201955243
                ],
                [
                    -0.8360306811298173,
                    -0.5401215629119622,
                    -0.09654738467277664
                ]
            ],
            "t": [
                [
                    2.460461730591029
                ],
                [
                    96.11590158330897
                ],
                [
                    356.7742262298035
                ]
            ],
            "panel": 0,
            "node": 19
        },
        {
            "name": "00_25",
            "type": "custom",
            "R": [
                [
                    -0.11281253207922451,
                    0.003853778557179469,
                    0.9936088168875647
                ],
                [
                    -0.526262362370532,
                    -0.848445744533578,
                    -0.05646011455044132
                ],
                [
                    0.8428055876405073,
                    -0.5292683317313576,
                    0.09774341138023
                ]
            ],
            "t": [
                [
                    -2.739373020427457
                ],
                [
                    100.88572545169313
                ],
                [
                    355.3559427706522
                ]
            ]
        },
        {
            "name": "00_30",
            "type": "custom",
            "R": [
                [
                    0.8330652020769675,
                    0.004455972777790915,
                    0.5531568614733662
                ],
                [
                    -0.33154320863135767,
                    -0.7964420244987315,
                    0.5057264106439229
                ],
                [
                    0.44281087373605144,
                    -0.6046984752080701,
                    -0.6620107885693023
                ]
            ],
            "t": [
                [
                    10.088579528011651
                ],
                [
                    66.58382637011508
                ],
                [
                    363.60032183025817
                ]
            ]
        },
        {
            "name": "00_31",
            "type": "custom",
            "R": [
                [
                    -0.8094820137654419,
                    0.05640705311399299,
                    -0.5844288782642765
                ],
                [
                    0.2728686692010744,
                    -0.8452111398878334,
                    -0.4595223807149592
                ],
                [
                    -0.5198861017171741,
                    -0.5314474323660849,
                    0.6687914980566184
                ]
            ],
            "t": [
                [
                    -13.680384808639468
                ],
                [
                    100.51446395952927
                ],
                [
                    355.92608839701103
                ]
            ]
        },
        {
            "name": "00_39",
            "type": "custom",
            "R": [
                [
                    -0.8943295980730513,
                    0.0013024064668271267,
                    0.44740683247788016
                ],
                [
                    -0.24293740503361425,
                    -0.8411507462752191,
                    -0.4831633670676775
                ],
                [
                    0.375707315933598,
                    -0.5407991547497432,
                    0.7525824120825138
                ]
            ],
            "t": [
                [
                    11.997661916383516
                ],
                [
                    96.13574812089016
                ],
                [
                    357.04796348948855
                ]
            ]
        }
    ]
}
    
"""



json_data_self = """
{
    "calibDataSource": "converted_custom_sequence",
    "cameras": [
        {
            "name": "00_19",
            "type": "custom",
            "resolution": [2304, 2664],
            "K": [[4565.195, 0.0, 2322.638], [0.0, 4564.824, 2726.120], [0.0, 0.0, 1.0]],
            "distCoef": [-0.145, 0.168, -0.0002, -0.0003, -0.062],
            "R": [
                [0.1373341854853676, 0.9898837178292, -0.03563069847352751],
                [0.531217490023328, -0.10396551201955243, -0.84083241529084],
                [-0.8360306811298173, 0.09654738467277664, -0.5401215629119622]
            ],
            "t": [[2468.7784586636285], [-268.88450545265334], [2736.0648632412617]],
            "panel": 0, "node": 19
        },
        {
            "name": "00_25",
            "type": "custom",
            "R": [
                [-0.11281253207922451, -0.9936088168875647, 0.003853778557179469],
                [-0.526262362370532, 0.05646011455044132, -0.848445744533578],
                [0.8428055876405073, -0.09774341138023, -0.5292683317313576]
            ],
            "t": [[-2467.126495694369], [263.15817299293695], [2736.8526838172506]]
        },
        {
            "name": "00_30",
            "type": "custom",
            "R": [
                [0.8330652020769675, -0.5531568614733662, 0.004455972777790915],
                [-0.33154320863135767, -0.5057264106439229, -0.7964420244987315],
                [0.44281087373605144, 0.6620107885693023, -0.6046984752080701]
            ],
            "t": [[-1473.3520530583755], [-2014.535692733654], [2728.537632332639]]
        },
        {
            "name": "00_31",
            "type": "custom",
            "R": [
                [-0.8094820137654419, 0.5844288782642765, 0.05640705311399299],
                [0.2728686692010744, 0.4595223807149592, -0.8452111398878334],
                [-0.5198861017171741, -0.6687914980566184, -0.5314474323660849]
            ],
            "t": [[1465.3975313608462], [1998.4690802943528], [2748.8362064166477]]
        },
        {
            "name": "00_39",
            "type": "custom",
            "R": [
                [-0.8943295980730513, -0.44740683247788016, 0.0013024064668271267],
                [-0.24293740503361425, 0.4831633670676775, -0.8411507462752191],
                [0.375707315933598, -0.7525824120825138, -0.5407991547497432]
            ],
            "t": [[-1000.6069868324321], [2276.265817495589], [2739.4026730337923]]
        }
    ]
}

"""

json_data_panoptic = """
{
	"calibDataSource": "160422_calib_tent13_norm",
	"cameras": [
		{
			"name": "00_00",
			"type": "hd",
			"resolution": [1920,1080],
			"K": [
				[1634.6,0,941.874],
				[0,1630.26,557.782],
				[0,0,1]
			],
			"distCoef": [-0.224939,0.206807,0.000164677,0.000641065,0.0176898],
			"R": [
				[0.1656794936,0.0336560618,-0.9856051821],
				[-0.09224101321,0.9955650135,0.01849052095],
				[0.9818563545,0.08784972047,0.1680491765]
			],
			"t": [
				[17.76193366],
				[126.741365],
				[286.3860507]
			],
			"panel": 0,
			"node": 0
		},
		{
			"name": "00_03",
			"type": "hd",
			"resolution": [1920,1080],
			"K": [
				[1397.21,0,950.563],
				[0,1394.06,565.694],
				[0,0,1]
			],
			"distCoef": [-0.286076,0.179787,-0.00024877,0.000216189,-0.0458576],
			"R": [
				[-0.6398574136,-0.02856086169,0.7679627383],
				[0.07251071636,0.992607153,0.09733054915],
				[-0.7650651516,0.1179632017,-0.6330560772]
			],
			"t": [
				[-12.82634653],
				[118.0983305],
				[285.3213573]
			],
			"panel": 0,
			"node": 3
		},
		{
			"name": "00_06",
			"type": "hd",
			"resolution": [1920,1080],
			"K": [
				[1397.06,0,951.324],
				[0,1393.61,548.802],
				[0,0,1]
			],
			"distCoef": [-0.285811,0.182134,-0.000208357,0.00114507,-0.0469778],
			"R": [
				[0.9526902996,-0.04878471384,-0.3000020746],
				[-0.1958615622,0.6562345106,-0.7286937048],
				[0.2324208285,0.752978299,0.6156332023]
			],
			"t": [
				[-8.934845719],
				[80.47056857],
				[380.5787661]
			],
			"panel": 0,
			"node": 6
		},
		{
			"name": "00_12",
			"type": "hd",
			"resolution": [1920,1080],
			"K": [
				[1396.3,0,964.516],
				[0,1393.06,563.848],
				[0,0,1]
			],
			"distCoef": [-0.2903,0.193774,-0.000299024,0.000664956,-0.058348],
			"R": [
				[-0.992641141,0.04264285975,-0.1133364535],
				[-0.01361196809,0.890718507,0.454351467],
				[0.1203257225,0.4525506908,-0.8835833819]
			],
			"t": [
				[21.30022365],
				[99.76498882],
				[330.9356912]
			],
			"panel": 0,
			"node": 12
		},
		{
			"name": "00_13",
			"type": "hd",
			"resolution": [1920,1080],
			"K": [
				[1591.24,0,937.462],
				[0,1587.6,563.669],
				[0,0,1]
			],
			"distCoef": [-0.23459,0.210043,-1.08788e-05,0.000501576,-0.0119359],
			"R": [
				[-0.2622312855,0.07402845689,-0.96216139],
				[-0.763225395,0.5942280278,0.2537322355],
				[0.5905266711,0.8008825373,-0.09932478135]
			],
			"t": [
				[-20.8800548],
				[58.96122399],
				[381.9527735]
			],
			"panel": 0,
			"node": 13
		},
        {
			"name": "00_23",
			"type": "hd",
			"resolution": [1920,1080],
			"K": [
				[1574.78,0,940.792],
				[0,1570.34,560.546],
				[0,0,1]
			],
			"distCoef": [-0.243717,0.20726,-0.000410568,0.000376085,-0.00713067],
			"R": [
				[0.4968169474,0.01226278331,0.8677687162],
				[0.1460941906,0.9844489522,-0.09755382099],
				[-0.8554702848,0.1752423598,0.4872994019]
			],
			"t": [
				[7.007243622],
				[101.9249307],
				[298.7701177]
			],
			"panel": 0,
			"node": 23
		}
    ]
}
"""

# --- 3. Process Data and Run ---

data = json.loads(json_data_panoptic)
extrinsics = []
names = []

for cam in data['cameras']:
    # 1. Get current R and Position (which is currently stored in t)
    R = np.array(cam['R'])
    C = np.array(cam['t']).flatten() # This is currently Position!
    
    # 2. Calculate the correct Translation Vector: t = -R * C
    # We reshape C to (3,1) for matrix multiplication
    t_correct = -R @ C
    
    # 3. Update the JSON entry
    # (Reshaping back to list of lists if that's your preferred format, 
    # or flat list depending on parser requirements. Your example uses list of lists)
    cam['t'] = t_correct.reshape(3, 1).tolist()

# Now save 'data' to a new JSON file and use THIS with MVGFormer
print("Conversion complete. 't' is now the translation vector.")

for cam in data['cameras']:
    # 1. Get R and t
    R = np.array(cam['R'])
    t = np.array(cam['t'])
    
    # 2. Construct 4x4 Extrinsic Matrix
    # Shape of t in JSON is (3, 1), so we flatten it to (3,) for assignment
    E = np.eye(4)
    E[:3, :3] = R
    E[:3, 3] = t.flatten() 
    
    extrinsics.append(E)
    names.append(cam.get('name', 'unknown'))

# 4. Visualize
# I slightly modified your function above to accept 'labels' so you can see "00_19" etc.
visualize_cameras(extrinsics, Title="Camera Calibration Layout", labels=names)

Conversion complete. 't' is now the translation vector.


### CONVERT FOR JSON DATA_SELF

In [2]:
import json
import numpy as np

# 1. Your Original "Self" Data
json_data_self = """
{
  "cameras": [
    {
      "name": "00_19",
      "type": "custom",
      "resolution": [
        2304,
        2664
      ],
      "K": [
        [
          2282.5976998,
          0.0,
          1161.319321
        ],
        [
          0.0,
          2282.412361,
          1363.060333
        ],
        [
          0.0,
          0.0,
          1.0
        ]
      ],
      "distCoef": [
        -0.14563766220819147,
        0.1689546054710371,
        -0.00024370898128271823,
        -0.0003999433188876148,
        -0.062989052568873
      ],
      "R": [
        [
          -0.1373341854853676,
          -0.03563069847352751,
          0.9898837178292
        ],
        [
          0.531217490023328,
          0.84083241529084,
          0.10396551201955243
        ],
        [
          -0.8360306811298173,
          0.5401215629119622,
          -0.09654738467277664
        ]
      ],
      "t": [
        [
          24.604617
        ],
        [
          -961.159016
        ],
        [
          -3567.742262
        ]
      ],
      "panel": 0,
      "node": 19
    },
    {
      "name": "00_25",
      "type": "custom",
      "resolution": [
        2304,
        2664
      ],
      "K": [
        [
          2277.034314,
          0.0,
          1121.647915
        ],
        [
          0.0,
          2276.839685,
          1333.498582
        ],
        [
          0.0,
          0.0,
          1.0
        ]
      ],
      "distCoef": [
        -0.133269627900226,
        0.12383338108439836,
        -0.00014747505783976278,
        -0.00018930144639310709,
        -0.007682486138965147
      ],
      "R": [
        [
          0.11281253207922451,
          0.003853778557179469,
          -0.9936088168875647
        ],
        [
          -0.526262362370532,
          0.848445744533578,
          -0.05646011455044132
        ],
        [
          0.8428055876405073,
          0.5292683317313576,
          0.09774341138023
        ]
      ],
      "t": [
        [
          -27.393730
        ],
        [
          -1008.857254
        ],
        [
          -3553.559428
        ]
      ],
      "panel": 0,
      "node": 25
    },
    {
      "name": "00_30",
      "type": "custom",
      "resolution": [
        2304,
        2664
      ],
      "K": [
        [
          2288.276488,
          0.0,
          1128.453708
        ],
        [
          0.0,
          2288.061192,
          1338.415642
        ],
        [
          0.0,
          0.0,
          1.0
        ]
      ],
      "distCoef": [
        -0.14421021184885766,
        0.16483567764342566,
        -0.0001152344579226653,
        -0.00020500562928971175,
        -0.06051264326424438
      ],
      "R": [
        [
          -0.8330652020769675,
          0.004455972777790915,
          -0.5531568614733662
        ],
        [
          -0.33154320863135767,
          0.7964420244987315,
          0.5057264106439229
        ],
        [
          0.44281087373605144,
          0.6046984752080701,
          -0.6620107885693023
        ]
      ],
      "t": [
        [
          100.885795
        ],
        [
          -665.838264
        ],
        [
          -3636.003218
        ]
      ],
      "panel": 0,
      "node": 30
    },
    {
      "name": "00_31",
      "type": "custom",
      "resolution": [
        2304,
        2664
      ],
      "K": [
        [
          2281.248624,
          0.0,
          1137.192555
        ],
        [
          0.0,
          2281.165084,
          1342.040428
        ],
        [
          0.0,
          0.0,
          1.0
        ]
      ],
      "distCoef": [
        -0.14210328353727048,
        0.1525500949106775,
        -1.540352967094957e-05,
        -0.00026252285885584186,
        -0.03889362548707034
      ],
      "R": [
        [
          0.8094820137654419,
          0.05640705311399299,
          0.5844288782642765
        ],
        [
          0.2728686692010744,
          0.8452111398878334,
          -0.4595223807149592
        ],
        [
          -0.5198861017171741,
          0.5314474323660849,
          0.6687914980566184
        ]
      ],
      "t": [
        [
          -136.803848
        ],
        [
          -1005.144640
        ],
        [
          -3559.260884
        ]
      ],
      "panel": 0,
      "node": 31
    },
    {
      "name": "00_39",
      "type": "custom",
      "resolution": [
        2304,
        2664
      ],
      "K": [
        [
          2281.275339,
          0.0,
          1135.784848
        ],
        [
          0.0,
          2281.111231,
          1321.627304
        ],
        [
          0.0,
          0.0,
          1.0
        ]
      ],
      "distCoef": [
        -0.13762141498673444,
        0.14458912949828817,
        -0.00013271762970704592,
        -0.0001550467908009401,
        -0.03542371634158656
      ],
      "R": [
        [
          0.8943295980730513,
          0.0013024064668271267,
          -0.44740683247788016
        ],
        [
          -0.24293740503361425,
          0.8411507462752191,
          -0.4831633670676775
        ],
        [
          0.375707315933598,
          0.5407991547497432,
          0.7525824120825138
        ]
      ],
      "t": [
        [
          119.976619
        ],
        [
          -961.357481
        ],
        [
          -3570.479635
        ]
      ],
      "panel": 0,
      "node": 39
    }
  ]
}
"""

data = json.loads(json_data_panoptic)
new_cameras = []

# --- 90 DEGREE SPIN MATRIX (Around Z-Axis) ---
# This keeps Z (height) unchanged but rotates X/Y.
# This should move the relative position of the origin (ground) to the right.
# Rotation: X -> Y, Y -> -X
R_spin = np.array([
    [ -1,  0,  0,  0],
    [ 0,  -1,  0,  0],
    [ 0,  0,  1, 0],
    [ 0,  0,  0,  1]
])


for cam in data['cameras']:
    # 1. Get Original Data
    R_raw = np.array(cam['R'])
    # Current 't' is Position (C) in mm
    C_raw = np.array(cam['t']).flatten()
    
    # 2. Scale Units (mm -> cm)
    C_cm = C_raw * 0.1
    
    # 3. Compute Translation Vector
    # t = -R * C
    t_vec = -R_raw @ C_cm
    
    # 4. Construct Extrinsic Matrix
    E_raw = np.eye(4)
    E_raw[:3, :3] = R_raw
    E_raw[:3, 3] = t_vec
    
    # 5. Apply the spin rotation
    E_final = E_raw @ R_spin.T
    
    # 6. Extract New R and t
    R_new = E_final[:3, :3]
    t_new = E_final[:3, 3]
    
    # 7. Update Dictionary
    cam_new = cam.copy()
    cam_new['R'] = R_new.tolist()
    cam_new['t'] = [[t_new[0]], [t_new[1]], [t_new[2]]]
    
    new_cameras.append(cam_new)

output_data = {
    "calibDataSource": "converted_self_to_panoptic_spin",
    "cameras": new_cameras
}

print(json.dumps(output_data, indent=4))
output_filename = "converted_calibration.json"
with open(output_filename, "w") as f:
    json.dump(output_data, f, indent=4)

print(f"Successfully saved converted calibration to {output_filename}")

json_data_self = json.dumps(output_data, indent=4)

{
    "calibDataSource": "converted_self_to_panoptic_spin",
    "cameras": [
        {
            "name": "00_00",
            "type": "hd",
            "resolution": [
                1920,
                1080
            ],
            "K": [
                [
                    1634.6,
                    0,
                    941.874
                ],
                [
                    0,
                    1630.26,
                    557.782
                ],
                [
                    0,
                    0,
                    1
                ]
            ],
            "distCoef": [
                -0.224939,
                0.206807,
                0.000164677,
                0.000641065,
                0.0176898
            ],
            "R": [
                [
                    -0.1656794936,
                    -0.0336560618,
                    -0.9856051821
                ],
                [
                    0.09224101321,
          

In [3]:
json_data_self = """{
  "cameras": [
    {
      "name": "00_19",
      "type": "custom",
      "resolution": [
        2304,
        2664
      ],
      "K": [
        [
          2282.5976998,
          0.0,
          1161.319321
        ],
        [
          0.0,
          2282.412361,
          1363.060333
        ],
        [
          0.0,
          0.0,
          1.0
        ]
      ],
      "distCoef": [
        -0.14563766220819147,
        0.1689546054710371,
        -0.00024370898128271823,
        -0.0003999433188876148,
        -0.062989052568873
      ],
      "R": [
        [
          -0.1373341854853676,
          -0.03563069847352751,
          0.9898837178292
        ],
        [
          0.531217490023328,
          0.84083241529084,
          0.10396551201955243
        ],
        [
          -0.8360306811298173,
          0.5401215629119622,
          -0.09654738467277664
        ]
      ],
      "t": [
        [
          24.604617
        ],
        [
          -961.159016
        ],
        [
          -3567.742262
        ]
      ],
      "panel": 0,
      "node": 19
    },
    {
      "name": "00_25",
      "type": "custom",
      "resolution": [
        2304,
        2664
      ],
      "K": [
        [
          2277.034314,
          0.0,
          1121.647915
        ],
        [
          0.0,
          2276.839685,
          1333.498582
        ],
        [
          0.0,
          0.0,
          1.0
        ]
      ],
      "distCoef": [
        -0.133269627900226,
        0.12383338108439836,
        -0.00014747505783976278,
        -0.00018930144639310709,
        -0.007682486138965147
      ],
      "R": [
        [
          0.11281253207922451,
          0.003853778557179469,
          -0.9936088168875647
        ],
        [
          -0.526262362370532,
          0.848445744533578,
          -0.05646011455044132
        ],
        [
          0.8428055876405073,
          0.5292683317313576,
          0.09774341138023
        ]
      ],
      "t": [
        [
          -27.393730
        ],
        [
          -1008.857254
        ],
        [
          -3553.559428
        ]
      ],
      "panel": 0,
      "node": 25
    },
    {
      "name": "00_30",
      "type": "custom",
      "resolution": [
        2304,
        2664
      ],
      "K": [
        [
          2288.276488,
          0.0,
          1128.453708
        ],
        [
          0.0,
          2288.061192,
          1338.415642
        ],
        [
          0.0,
          0.0,
          1.0
        ]
      ],
      "distCoef": [
        -0.14421021184885766,
        0.16483567764342566,
        -0.0001152344579226653,
        -0.00020500562928971175,
        -0.06051264326424438
      ],
      "R": [
        [
          -0.8330652020769675,
          0.004455972777790915,
          -0.5531568614733662
        ],
        [
          -0.33154320863135767,
          0.7964420244987315,
          0.5057264106439229
        ],
        [
          0.44281087373605144,
          0.6046984752080701,
          -0.6620107885693023
        ]
      ],
      "t": [
        [
          100.885795
        ],
        [
          -665.838264
        ],
        [
          -3636.003218
        ]
      ],
      "panel": 0,
      "node": 30
    },
    {
      "name": "00_31",
      "type": "custom",
      "resolution": [
        2304,
        2664
      ],
      "K": [
        [
          2281.248624,
          0.0,
          1137.192555
        ],
        [
          0.0,
          2281.165084,
          1342.040428
        ],
        [
          0.0,
          0.0,
          1.0
        ]
      ],
      "distCoef": [
        -0.14210328353727048,
        0.1525500949106775,
        -1.540352967094957e-05,
        -0.00026252285885584186,
        -0.03889362548707034
      ],
      "R": [
        [
          0.8094820137654419,
          0.05640705311399299,
          0.5844288782642765
        ],
        [
          0.2728686692010744,
          0.8452111398878334,
          -0.4595223807149592
        ],
        [
          -0.5198861017171741,
          0.5314474323660849,
          0.6687914980566184
        ]
      ],
      "t": [
        [
          -136.803848
        ],
        [
          -1005.144640
        ],
        [
          -3559.260884
        ]
      ],
      "panel": 0,
      "node": 31
    },
    {
      "name": "00_39",
      "type": "custom",
      "resolution": [
        2304,
        2664
      ],
      "K": [
        [
          2281.275339,
          0.0,
          1135.784848
        ],
        [
          0.0,
          2281.111231,
          1321.627304
        ],
        [
          0.0,
          0.0,
          1.0
        ]
      ],
      "distCoef": [
        -0.13762141498673444,
        0.14458912949828817,
        -0.00013271762970704592,
        -0.0001550467908009401,
        -0.03542371634158656
      ],
      "R": [
        [
          0.8943295980730513,
          0.0013024064668271267,
          -0.44740683247788016
        ],
        [
          -0.24293740503361425,
          0.8411507462752191,
          -0.4831633670676775
        ],
        [
          0.375707315933598,
          0.5407991547497432,
          0.7525824120825138
        ]
      ],
      "t": [
        [
          119.976619
        ],
        [
          -961.357481
        ],
        [
          -3570.479635
        ]
      ],
      "panel": 0,
      "node": 39
    }
  ]
}"""

In [36]:
json_data_converted = """ {
    "cameras": [
        {
            "name": "00_19",
            "type": "custom",
            "resolution": [
                2304,
                2664
            ],
            "K": [
                [
                    2282.5976998,
                    0.0,
                    1161.319321
                ],
                [
                    0.0,
                    2282.412361,
                    1363.060333
                ],
                [
                    0.0,
                    0.0,
                    1.0
                ]
            ],
            "distCoef": [
                -0.14563766220819147,
                0.1689546054710371,
                -0.00024370898128271823,
                -0.0003999433188876148,
                -0.062989052568873
            ],
            "R": [
                [
                    -0.1373341854853676,
                    -0.03563069847352751,
                    -0.9898837178292
                ],
                [
                    -0.531217490023328,
                    -0.84083241529084,
                    0.10396551201955243
                ],
                [
                    -0.8360306811298173,
                    0.5401215629119622,
                    0.09654738467277664
                ]
            ],
            "t": [
                [
                    -2.4604617
                ],
                [
                    -96.1159016
                ],
                [
                    356.77422620000004
                ]
            ],
            "panel": 0,
            "node": 19
        },
        {
            "name": "00_25",
            "type": "custom",
            "resolution": [
                2304,
                2664
            ],
            "K": [
                [
                    2277.034314,
                    0.0,
                    1121.647915
                ],
                [
                    0.0,
                    2276.839685,
                    1333.498582
                ],
                [
                    0.0,
                    0.0,
                    1.0
                ]
            ],
            "distCoef": [
                -0.133269627900226,
                0.12383338108439836,
                -0.00014747505783976278,
                -0.00018930144639310709,
                -0.007682486138965147
            ],
            "R": [
                [
                    0.11281253207922451,
                    0.003853778557179469,
                    0.9936088168875647
                ],
                [
                    0.526262362370532,
                    -0.848445744533578,
                    -0.05646011455044132
                ],
                [
                    0.8428055876405073,
                    0.5292683317313576,
                    -0.09774341138023
                ]
            ],
            "t": [
                [
                    2.739373
                ],
                [
                    -100.8857254
                ],
                [
                    355.3559428
                ]
            ],
            "panel": 0,
            "node": 25
        },
        {
            "name": "00_30",
            "type": "custom",
            "resolution": [
                2304,
                2664
            ],
            "K": [
                [
                    2288.276488,
                    0.0,
                    1128.453708
                ],
                [
                    0.0,
                    2288.061192,
                    1338.415642
                ],
                [
                    0.0,
                    0.0,
                    1.0
                ]
            ],
            "distCoef": [
                -0.14421021184885766,
                0.16483567764342566,
                -0.0001152344579226653,
                -0.00020500562928971175,
                -0.06051264326424438
            ],
            "R": [
                [
                    -0.8330652020769675,
                    0.004455972777790915,
                    0.5531568614733662
                ],
                [
                    0.33154320863135767,
                    -0.7964420244987315,
                    0.5057264106439229
                ],
                [
                    0.44281087373605144,
                    0.6046984752080701,
                    0.6620107885693023
                ]
            ],
            "t": [
                [
                    -10.0885795
                ],
                [
                    -66.58382639999999
                ],
                [
                    363.60032179999996
                ]
            ],
            "panel": 0,
            "node": 30
        },
        {
            "name": "00_31",
            "type": "custom",
            "resolution": [
                2304,
                2664
            ],
            "K": [
                [
                    2281.248624,
                    0.0,
                    1137.192555
                ],
                [
                    0.0,
                    2281.165084,
                    1342.040428
                ],
                [
                    0.0,
                    0.0,
                    1.0
                ]
            ],
            "distCoef": [
                -0.14210328353727048,
                0.1525500949106775,
                -1.540352967094957e-05,
                -0.00026252285885584186,
                -0.03889362548707034
            ],
            "R": [
                [
                    0.8094820137654419,
                    0.05640705311399299,
                    -0.5844288782642765
                ],
                [
                    -0.2728686692010744,
                    -0.8452111398878334,
                    -0.4595223807149592
                ],
                [
                    -0.5198861017171741,
                    0.5314474323660849,
                    -0.6687914980566184
                ]
            ],
            "t": [
                [
                    13.680384799999999
                ],
                [
                    -100.514464
                ],
                [
                    355.92608839999997
                ]
            ],
            "panel": 0,
            "node": 31
        },
        {
            "name": "00_39",
            "type": "custom",
            "resolution": [
                2304,
                2664
            ],
            "K": [
                [
                    2281.275339,
                    0.0,
                    1135.784848
                ],
                [
                    0.0,
                    2281.111231,
                    1321.627304
                ],
                [
                    0.0,
                    0.0,
                    1.0
                ]
            ],
            "distCoef": [
                -0.13762141498673444,
                0.14458912949828817,
                -0.00013271762970704592,
                -0.0001550467908009401,
                -0.03542371634158656
            ],
            "R": [
                [
                    0.8943295980730513,
                    0.0013024064668271267,
                    0.44740683247788016
                ],
                [
                    0.24293740503361425,
                    -0.8411507462752191,
                    -0.4831633670676775
                ],
                [
                    0.375707315933598,
                    0.5407991547497432,
                    -0.7525824120825138
                ]
            ],
            "t": [
                [
                    -11.9976619
                ],
                [
                    -96.1357481
                ],
                [
                    357.04796350000004
                ]
            ],
            "panel": 0,
            "node": 39
        }
    ]
}"""

### Visualize panoptic

In [37]:
import json
import numpy as np
import plotly.graph_objects as go

def visualize_cameras(extrinsic_matrices, Title="", scale=200.0, labels=None):
    fig = go.Figure()
    all_x, all_y, all_z = [], [], []
    cam_centers = []
    
    if labels is None:
        plot_labels = [f"Cam {i}" for i in range(len(extrinsic_matrices))]
    else:
        plot_labels = labels

    for i in range(len(extrinsic_matrices)):
        E = extrinsic_matrices[i]
        R = E[:3, :3]
        t = E[:3, 3]
        
        # Camera center in world coordinates: C = -R^T * t
        C = -R.T @ t
        
        # Camera-to-World matrix
        C_to_W = np.eye(4)
        C_to_W[:3, :3] = R.T  # Inverse of rotation
        C_to_W[:3, 3] = C     # Camera center
        cam_pos = C_to_W[:3, 3]
        cam_centers.append(cam_pos)
        
        # Frustum Calculation
        s = scale
        corners_cam = np.array([
            [0, 0, 0],          
            [-s, -s, s*1.5],    
            [s, -s, s*1.5],     
            [s, s, s*1.5],      
            [-s, s, s*1.5],     
        ])
        
        corners_homo = np.hstack([corners_cam, np.ones((5, 1))])
        p = (C_to_W @ corners_homo.T).T[:, :3]
        
        indices = [0, 1, 2, 0, 2, 3, 0, 3, 4, 0, 4, 1, 1, 2, 3, 4, 1]
        for idx in indices:
            all_x.append(p[idx, 0]); all_y.append(p[idx, 1]); all_z.append(p[idx, 2])
        all_x.append(None); all_y.append(None); all_z.append(None)

    fig.add_trace(go.Scatter3d(
        x=all_x, y=all_y, z=all_z,
        mode='lines',
        line=dict(color='rgba(100, 100, 255, 0.6)', width=2),
        name='Frustums',
        hoverinfo='none'
    ))

    cam_centers = np.array(cam_centers)
    fig.add_trace(go.Scatter3d(
        x=cam_centers[:, 0], y=cam_centers[:, 1], z=cam_centers[:, 2],
        mode='markers+text',
        marker=dict(size=6, color=np.arange(len(extrinsic_matrices)), colorscale='Viridis', opacity=0.8),
        text=plot_labels,
        textposition="top center",
        name='Camera Positions'
    ))

    fig.add_trace(go.Scatter3d(
        x=[0], y=[0], z=[0],
        mode='markers',
        marker=dict(size=8, color='red', symbol='cross'),
        name='World Origin'
    ))

    fig.update_layout(
        scene=dict(xaxis_title='X (cm)', yaxis_title='Y (cm)', zaxis_title='Z (cm)', aspectmode='data'),
        margin=dict(l=0, r=0, b=0, t=40),
        title=f"{Title}"
    )
    fig.show()

# Load Panoptic data
data = json.loads(json_data_converted)
extrinsics = []
names = []

# NO CONVERSION NEEDED - t is already the translation vector in Panoptic format
for cam in data['cameras']:
    R = np.array(cam['R'])
    t = np.array(cam['t']).flatten()  # Already correct!
    
    E = np.eye(4)
    E[:3, :3] = R
    E[:3, 3] = t
    
    extrinsics.append(E)
    names.append(cam.get('name', 'unknown'))

visualize_cameras(extrinsics, Title="self Camera Layout", scale=20.0, labels=names)

## adjust vis correct data

In [29]:
import json
import numpy as np
import plotly.graph_objects as go
import plotly.offline as pyo

# 1. INITIALIZE NOTEBOOK MODE (Critical for Jupyter)
pyo.init_notebook_mode(connected=True)

# ---------------------------------------------------------
# VISUALIZATION FUNCTION (Restored Original Style)
# ---------------------------------------------------------
def visualize_cameras(extrinsic_matrices, Title="", scale=200.0, labels=None):
    fig = go.Figure()
    all_x, all_y, all_z = [], [], []
    cam_centers = []
    
    if labels is None:
        plot_labels = [f"Cam {i}" for i in range(len(extrinsic_matrices))]
    else:
        plot_labels = labels

    for i in range(len(extrinsic_matrices)):
        E = extrinsic_matrices[i]
        R = E[:3, :3]
        t = E[:3, 3]
        
        # Camera center in world coordinates: C = -R^T * t
        C = -R.T @ t
        
        # Camera-to-World matrix
        C_to_W = np.eye(4)
        C_to_W[:3, :3] = R.T  # Inverse of rotation
        C_to_W[:3, 3] = C     # Camera center
        cam_pos = C_to_W[:3, 3]
        cam_centers.append(cam_pos)
        
        # Frustum Calculation
        s = scale
        corners_cam = np.array([
            [0, 0, 0],          
            [-s, -s, s*1.5],    
            [s, -s, s*1.5],     
            [s, s, s*1.5],      
            [-s, s, s*1.5],     
        ])
        
        corners_homo = np.hstack([corners_cam, np.ones((5, 1))])
        p = (C_to_W @ corners_homo.T).T[:, :3]
        
        indices = [0, 1, 2, 0, 2, 3, 0, 3, 4, 0, 4, 1, 1, 2, 3, 4, 1]
        for idx in indices:
            all_x.append(p[idx, 0]); all_y.append(p[idx, 1]); all_z.append(p[idx, 2])
        all_x.append(None); all_y.append(None); all_z.append(None)

    # Frustums Trace
    fig.add_trace(go.Scatter3d(
        x=all_x, y=all_y, z=all_z,
        mode='lines',
        line=dict(color='rgba(100, 100, 255, 0.6)', width=2),
        name='Frustums',
        hoverinfo='none'
    ))

    # Camera Positions Trace
    cam_centers = np.array(cam_centers)
    fig.add_trace(go.Scatter3d(
        x=cam_centers[:, 0], y=cam_centers[:, 1], z=cam_centers[:, 2],
        mode='markers+text',
        marker=dict(size=6, color=np.arange(len(extrinsic_matrices)), colorscale='Viridis', opacity=0.8),
        text=plot_labels,
        textposition="top center",
        name='Camera Positions'
    ))

    # World Origin Trace
    fig.add_trace(go.Scatter3d(
        x=[0], y=[0], z=[0],
        mode='markers',
        marker=dict(size=8, color='red', symbol='cross'),
        name='World Origin'
    ))

    fig.update_layout(
        scene=dict(xaxis_title='X (cm)', yaxis_title='Y (cm)', zaxis_title='Z (cm)', aspectmode='data'),
        margin=dict(l=0, r=0, b=0, t=40),
        title=f"{Title}"
    )
    
    # 2. STANDARD SHOW METHOD (Let init_notebook_mode handle the renderer)
    fig.show()

# ---------------------------------------------------------
# DATA CONVERSION LOGIC
# ---------------------------------------------------------
# Assuming 'json_data_self' is already defined in your notebook
data = json.loads(json_data_self)

extrinsics = []
names = []
converted_cameras = []

# TRANSFORMATION MATRICES
R_fix_axes = np.array([
    [1,  0,  0],
    [0, -1,  0],
    [0,  0, -1]
])

R_z_flip = np.array([
    [-1,  0,  0],
    [ 0, -1,  0],
    [ 0,  0,  1]
])

R_global = R_z_flip @ R_fix_axes

for cam in data['cameras']:
    R_raw = np.array(cam['R'])
    t_raw = np.array(cam['t']).flatten()
    
    # Unit Conversion (mm -> cm)
    t_cm = t_raw / 10.0
    
    # Apply Orientation Fixes
    R_new = R_global @ R_raw
    t_new = R_global @ t_cm
    
    R_sideways = np.array([
    [-1,  0,  0, 0],  
    [0,  -1,  0, 0],  
    [0, 0,  1, 0], 
    [0,  0,  0, 1]
])

    # Create 4x4 Matrix
    E = np.eye(4)
    E[:3, :3] = R_new
    E[:3, 3] = t_new
    
    E = E @ R_sideways

    extrinsics.append(E)
    names.append(cam.get('name', 'unknown'))
    
    new_cam = cam.copy()
    new_cam['t'] = [[x] for x in t_new]
    new_cam['R'] = R_new.tolist()
    converted_cameras.append(new_cam)

# Run Visualization
visualize_cameras(extrinsics, Title="Corrected Self Data (Panoptic Style)", scale=15.0, labels=names)

### adjust visualization of "self" calib with the "panoptic" visualization

In [22]:
import json
import numpy as np
import plotly.graph_objects as go

# --- Visualization Function (No changes needed) ---
def visualize_cameras(extrinsic_matrices, Title="", scale=20.0, labels=None):
    fig = go.Figure()
    all_x, all_y, all_z = [], [], []
    cam_centers = []
    
    if labels is None:
        plot_labels = [f"Cam {i}" for i in range(len(extrinsic_matrices))]
    else:
        plot_labels = labels

    for i in range(len(extrinsic_matrices)):
        E = extrinsic_matrices[i]
        R = E[:3, :3]
        t = E[:3, 3]
        C = -R.T @ t
        
        C_to_W = np.eye(4)
        C_to_W[:3, :3] = R.T
        C_to_W[:3, 3] = C
        cam_centers.append(C)
        
        s = scale
        corners_cam = np.array([
            [0, 0, 0], [-s, -s, s*1.5], [s, -s, s*1.5], [s, s, s*1.5], [-s, s, s*1.5],
        ])
        corners_homo = np.hstack([corners_cam, np.ones((5, 1))])
        p = (C_to_W @ corners_homo.T).T[:, :3]
        
        indices = [0, 1, 2, 0, 2, 3, 0, 3, 4, 0, 4, 1, 1, 2, 3, 4, 1]
        for idx in indices:
            all_x.append(p[idx, 0]); all_y.append(p[idx, 1]); all_z.append(p[idx, 2])
        all_x.append(None); all_y.append(None); all_z.append(None)

    fig.add_trace(go.Scatter3d(
        x=all_x, y=all_y, z=all_z, mode='lines',
        line=dict(color='rgba(100, 100, 255, 0.6)', width=2), name='Frustums', hoverinfo='none'
    ))
    cam_centers = np.array(cam_centers)
    fig.add_trace(go.Scatter3d(
        x=cam_centers[:, 0], y=cam_centers[:, 1], z=cam_centers[:, 2],
        mode='markers+text', marker=dict(size=6, color=np.arange(len(extrinsic_matrices)), colorscale='Viridis', opacity=0.8),
        text=plot_labels, textposition="top center", name='Camera Positions'
    ))
    fig.add_trace(go.Scatter3d(
        x=[0], y=[0], z=[0], mode='markers',
        marker=dict(size=8, color='red', symbol='cross'), name='World Origin'
    ))
    fig.update_layout(
        scene=dict(xaxis_title='X (Height?)', yaxis_title='Y', zaxis_title='Z', aspectmode='data'),
        margin=dict(l=0, r=0, b=0, t=40), title=f"{Title}"
    )
    fig.show()

# --- Main Processing Code ---
# --- Main Processing Code ---
data_self = json.loads(json_data_self)
extrinsics_self = []
names_self = []

# ROTATION CONFIGURATION
# Standard Side View: Map Height (Old Z) -> X, Y -> Y, Depth -> Z
# (Adjust this depending on if you want the wall on the left or right)
R_sideways = np.array([
    [0,  0,  1, 0],  
    [0,  1,  0, 0],  
    [-1, 0,  0, 0], 
    [0,  0,  0, 1]
])

for cam in data_self['cameras']:
    R = np.array(cam['R'])
    t_raw = np.array(cam['t']).flatten() # This is the Translation Vector (t), not Position (C)
    
    # 1. Scale mm -> cm (Scale t first)
    t_cm = t_raw * 0.1
    
    # 2. Calculate Real World Position (Camera Center)
    # Formula: C = -R_transpose * t
    C_cm = -R.T @ t_cm
    
    # 3. Build Extrinsic Matrix for Visualization
    # We need to construct the matrix that places the camera at C_cm
    E_raw = np.eye(4)
    E_raw[:3, :3] = R
    E_raw[:3, 3] = t_cm # The matrix usually stores t, not C
    
    # 4. Apply Side-View Rotation
    # We transform the entire camera pose into the new coordinate system
    E_final = E_raw @ R_sideways.T
    
    extrinsics_self.append(E_final)
    names_self.append(cam.get('name', 'unknown'))

visualize_cameras(extrinsics_self, Title="Self Calibration (Corrected Stage View)", scale=20.0, labels=names_self)

### Convert "self" calib to "panoptic" calib

In [30]:
import json
import numpy as np
import plotly.graph_objects as go
import plotly.offline as pyo

# 1. INITIALIZE NOTEBOOK MODE
pyo.init_notebook_mode(connected=True)

# ---------------------------------------------------------
# VISUALIZATION FUNCTION
# ---------------------------------------------------------
def visualize_cameras(extrinsic_matrices, Title="", scale=200.0, labels=None):
    fig = go.Figure()
    all_x, all_y, all_z = [], [], []
    cam_centers = []
    
    if labels is None:
        plot_labels = [f"Cam {i}" for i in range(len(extrinsic_matrices))]
    else:
        plot_labels = labels

    for i in range(len(extrinsic_matrices)):
        E = extrinsic_matrices[i]
        R = E[:3, :3]
        t = E[:3, 3]
        
        # Camera center in world coordinates: C = -R^T * t
        C = -R.T @ t
        
        # Camera-to-World matrix
        C_to_W = np.eye(4)
        C_to_W[:3, :3] = R.T
        C_to_W[:3, 3] = C
        cam_pos = C_to_W[:3, 3]
        cam_centers.append(cam_pos)
        
        # Frustum Calculation
        s = scale
        corners_cam = np.array([
            [0, 0, 0],          
            [-s, -s, s*1.5],    
            [s, -s, s*1.5],     
            [s, s, s*1.5],      
            [-s, s, s*1.5],     
        ])
        
        corners_homo = np.hstack([corners_cam, np.ones((5, 1))])
        p = (C_to_W @ corners_homo.T).T[:, :3]
        
        indices = [0, 1, 2, 0, 2, 3, 0, 3, 4, 0, 4, 1, 1, 2, 3, 4, 1]
        for idx in indices:
            all_x.append(p[idx, 0]); all_y.append(p[idx, 1]); all_z.append(p[idx, 2])
        all_x.append(None); all_y.append(None); all_z.append(None)

    # Frustums
    fig.add_trace(go.Scatter3d(
        x=all_x, y=all_y, z=all_z,
        mode='lines',
        line=dict(color='rgba(100, 100, 255, 0.6)', width=2),
        name='Frustums',
        hoverinfo='none'
    ))

    # Camera Positions
    cam_centers = np.array(cam_centers)
    fig.add_trace(go.Scatter3d(
        x=cam_centers[:, 0], y=cam_centers[:, 1], z=cam_centers[:, 2],
        mode='markers+text',
        marker=dict(size=6, color=np.arange(len(extrinsic_matrices)), colorscale='Viridis', opacity=0.8),
        text=plot_labels,
        textposition="top center",
        name='Camera Positions'
    ))

    # Origin
    fig.add_trace(go.Scatter3d(
        x=[0], y=[0], z=[0],
        mode='markers',
        marker=dict(size=8, color='red', symbol='cross'),
        name='World Origin'
    ))

    fig.update_layout(
        scene=dict(xaxis_title='X (cm)', yaxis_title='Y (cm)', zaxis_title='Z (cm)', aspectmode='data'),
        margin=dict(l=0, r=0, b=0, t=40),
        title=f"{Title}"
    )
    
    fig.show()

# ---------------------------------------------------------
# DATA CONVERSION LOGIC
# ---------------------------------------------------------
# Ensure 'json_data_self' is loaded before running this cell
data = json.loads(json_data_self)

extrinsics = []
names = []
converted_cameras = []

# TRANSFORMATION MATRICES
R_fix_axes = np.array([
    [1,  0,  0],
    [0, -1,  0],
    [0,  0, -1]
])

R_z_flip = np.array([
    [-1,  0,  0],
    [ 0, -1,  0],
    [ 0,  0,  1]
])

# Defines the "Sideways" flip (Rotate 180 around Z) you added in logic
R_sideways_4x4 = np.array([
    [-1,  0,  0, 0],  
    [ 0, -1,  0, 0],  
    [ 0,  0,  1, 0], 
    [ 0,  0,  0, 1]
])

# Global R calculation
R_global = R_z_flip @ R_fix_axes

for cam in data['cameras']:
    R_raw = np.array(cam['R'])
    t_raw = np.array(cam['t']).flatten()
    
    # 1. Unit Conversion (mm -> cm)
    t_cm = t_raw / 10.0
    
    # 2. Apply Initial Orientation Fixes
    R_temp = R_global @ R_raw
    t_temp = R_global @ t_cm
    
    # 3. Create 4x4 Matrix
    E_temp = np.eye(4)
    E_temp[:3, :3] = R_temp
    E_temp[:3, 3] = t_temp
    
    # 4. Apply the Sideways flip (E_final = E_temp * R_sideways)
    # Note: Applying matrix on the right rotates the camera frame itself
    E_final = E_temp @ R_sideways_4x4

    # Extract new R and t from the final 4x4 matrix
    R_final = E_final[:3, :3]
    t_final = E_final[:3, 3]
    
    # Store for visualization
    extrinsics.append(E_final)
    names.append(cam.get('name', 'unknown'))
    
    # Store for JSON Export
    new_cam = cam.copy()
    new_cam['t'] = [[x] for x in t_final] # Panoptic expects list of lists [[x],[y],[z]]
    new_cam['R'] = R_final.tolist()
    converted_cameras.append(new_cam)

# ---------------------------------------------------------
# EXECUTE & SAVE
# ---------------------------------------------------------

# 1. Visualize
visualize_cameras(extrinsics, Title="Corrected Self Data (Final)", scale=15.0, labels=names)

# 2. Save to JSON File
final_json_structure = {"cameras": converted_cameras}
filename = "converted_camera_calibration.json"

with open(filename, 'w') as f:
    json.dump(final_json_structure, f, indent=4)

print(f"\nSuccessfully converted {len(converted_cameras)} cameras.")
print(f"File saved as: {filename}")


Successfully converted 5 cameras.
File saved as: converted_camera_calibration.json
